# Data loading, featurization & round-trip

**One-time build → push → cached load.** This notebook builds QM9 and ZINC-250k from source,
runs the vocab-independent **sanitize + round-trip** filter, pushes the surviving
**canonical SMILES + targets** to the Hub as
[`nico8771/qm9_clean`](https://huggingface.co/datasets/nico8771/qm9_clean) /
[`nico8771/zinc_clean`](https://huggingface.co/datasets/nico8771/zinc_clean), then demonstrates
re-loading from the Hub and building a `MoleculeDataset`. Dense graph tensors `(X, E)` are
rebuilt lazily from SMILES — never stored.

- **§1–§6** build from source (`use_cache=False`) and inspect (round-trip, drop stats, target norms).
- **§7** pushes the cleaned datasets to the Hub (needs a write `HF_TOKEN`).
- **§8** re-loads from the Hub (`use_cache=True`, public/token-free) and builds the training dataset.

Downstream (training) just calls `load_qm9()` / `load_zinc()` with `use_cache=True` to pull these.

- **QM9** — neutral SMILES + DFT targets (`mu, alpha, homo, lumo, gap, cv`); round-trip *report-only*.
- **ZINC** — keeps `N+`/`O-`; `charge_aware=True`; round-trip filter *applied*; targets (`logP, qed, SAS`) recomputed via RDKit.

## Setup

In [1]:
import os

REPO = "flow-matching-molecules"
if not os.path.isdir(REPO):
    !git clone https://github.com/Nico-Conti/flow-matching-molecules.git
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else ".")

!pip install -q uv
!uv pip install --system -q -e .
print("cwd:", os.getcwd())

cwd: /content/flow-matching-molecules


## 1 · Load QM9

In [ ]:
import numpy as np
from dataset.qm9 import load_qm9
from dataset.zinc import load_zinc
from dataset.torch_dataset import MoleculeDataset, collate_dense

LIMIT = None   # set None for the full datasets

# One-time build from source (use_cache=False forces the sanitize/round-trip pass).
# §7 pushes the result to the Hub; §8 then re-loads it with use_cache=True (no rebuild).
qm9 = load_qm9(local_dir="data", limit=LIMIT, use_cache=False)
print("QM9 :", qm9["ds"].num_rows, "molecules | targets", qm9["targets"])
print("  stats:", qm9["stats"])

## 2 · Load ZINC-250k

In [ ]:
# One-time build from source (this is the slow ZINC sanitize pass).
zinc = load_zinc(local_dir="data", limit=LIMIT, use_cache=False)
print("ZINC:", zinc["ds"].num_rows, "molecules | targets", zinc["targets"])
print("  stats:", zinc["stats"])

## 3 · Featurizer round-trip checks
`smiles -> (X, E) -> smiles` should reproduce the canonical molecule.

In [ ]:
from rdkit import Chem
from dataset.featurize import smiles_to_tensor, tensor_to_mol, QM9_ATOMS, ZINC_ATOMS

def canon(s):
    return Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False)

def check(s, vocab, charge_aware):
    X, E = smiles_to_tensor(s, atom_vocab=vocab, charge_aware=charge_aware)
    m = tensor_to_mol(X, E, atom_vocab=vocab)
    got = Chem.MolToSmiles(m, isomericSmiles=False) if m is not None else None
    print(f"  {s:22s} -> {str(got):22s} match={m is not None and got == canon(s)}")

print("QM9 (charge_aware, neutral):")
for s in ["CCO", "CC(=O)O", "c1ccncc1", "N#Cc1ccccc1"]:
    check(s, QM9_ATOMS, charge_aware=True)
print("ZINC (charge-aware):")
for s in ["CC[N+](C)(C)C", "CC(=O)[O-]", "Clc1ccccc1", "c1ccc2ccccc2c1"]:
    check(s, ZINC_ATOMS, charge_aware=True)

## 4 · Round-trip drop summary
`kept` = round-trips exactly; `kept_no_roundtrip` = featurized but not gated (QM9);
`drop_*` = excluded (vocab / round-trip / parse).

In [ ]:
def show(name, stats):
    counts = {k: v for k, v in stats.items() if k.startswith(("drop_", "kept"))}
    if not counts:                       # loaded from the Hub cache (no drop breakdown)
        print(f"{name}: {stats}")
        return
    n = sum(counts.values())
    kept = counts.get("kept", 0) + counts.get("kept_no_roundtrip", 0)
    print(f"{name}: {kept:,}/{n:,} usable ({kept/n:.2%})")
    for k, v in sorted(counts.items()):
        print(f"    {k:20s} {v:>6,} ({v/n:.2%})")

show("QM9 ", qm9["stats"])
print()
show("ZINC", zinc["stats"])

## 6 · Targets present & per-property normalization stats
Confirms `y` is aligned (one row per usable molecule) and has no missing values,
then reports mean/std per property — the conditioning-normalization stats for training.

In [ ]:
def target_report(name, d):
    y = np.asarray(d["ds"]["y"])          # (N, T) — the y column as an array
    assert y.shape[0] == d["ds"].num_rows, f"{name}: y rows {y.shape[0]} != #mols {d['ds'].num_rows}"
    n_nan = int(np.isnan(y).sum())
    print(f"{name}: y {y.shape} | targets {d['targets']} | NaNs {n_nan}")
    for j, t in enumerate(d["targets"]):
        c = y[:, j]
        print(f"    {t:6s} mean={c.mean():+.4f}  std={c.std():.4f}  min={c.min():+.4f}  max={c.max():+.4f}")
    return {t: (float(y[:, j].mean()), float(y[:, j].std())) for j, t in enumerate(d["targets"])}

norm_stats = {"qm9": target_report("QM9 ", qm9), "zinc": target_report("ZINC", zinc)}
norm_stats

## 5 · Dataset + padded batch
`MoleculeDataset` + `collate_dense` pad variable-N graphs and emit a node mask — training-ready.

## 7 · Push cleaned datasets to the Hub

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()                        # your .env with HF_TOKEN (write token)
assert os.environ.get("HF_TOKEN"), "set HF_TOKEN in .env before pushing"

from dataset.qm9 import push_qm9
from dataset.zinc import push_zinc

push_qm9(qm9)                         # -> nico8771/qm9_clean  (data + dataset card)
push_zinc(zinc)                       # -> nico8771/zinc_clean (data + dataset card)
print("pushed both datasets")

## 8 · Load from the Hub & build the training dataset

Re-loads `nico8771/qm9_clean` from the Hub (`use_cache=True`, no token), wraps it in
`MoleculeDataset` (lazy `smiles -> (X, E)`), and splits **train/val/test in code** with a fixed
seed — the Hub holds one unsplit corpus, so the split lives here, not in the cached artifact.

In [ ]:
from torch.utils.data import DataLoader, random_split
import torch

# Re-load from the Hub (use_cache=True -> pulls nico8771/qm9_clean, public/token-free).
qm9 = load_qm9(use_cache=True)
full = MoleculeDataset.from_loader(qm9)          # lazy smiles -> (X, E)

# We store one unsplit corpus, so train/val/test is split here, in code, with a fixed seed.
n = len(full); n_test = int(0.1 * n); n_val = int(0.1 * n)
g = torch.Generator().manual_seed(0)
train, val, test = random_split(full, [n - n_val - n_test, n_val, n_test], generator=g)
print(f"loaded {n:,} from Hub | split train={len(train):,} val={len(val):,} test={len(test):,}")

loader = DataLoader(train, batch_size=4, shuffle=True, collate_fn=collate_dense)
print({k: tuple(v.shape) for k, v in next(iter(loader)).items()})